# Recursive Wikipedia category download (example: Finance in India)

This notebook walks the English Wikipedia **category tree** starting from [Category:Finance in India](https://en.wikipedia.org/wiki/Category:Finance_in_India), collects unique article titles, downloads plain text, writes **one `.txt` file per page**, applies light **cleaning**, and prints a **download report**.

You can **exclude entire subcategory branches** (e.g. skip *Indian people in finance* and everything nested under it) by editing `EXCLUDED_SUBCATEGORIES` below. Pages listed only under excluded branches are not collected; pages that also appear in a non-excluded category are still collected when first seen.

## Configuration

In [ ]:
from __future__ import annotations

from pathlib import Path

# Root category (with or without "Category:" prefix)
ROOT_CATEGORY = "Finance in India"

# Subcategory names to skip entirely (no traversal, no articles from that subtree).
# Match is case-insensitive; underscores vs spaces are normalized.
EXCLUDED_SUBCATEGORIES = [
    "Indian people in finance",
    # "Microfinance in India",
]

LANG = "en"
API_URL = f"https://{LANG}.wikipedia.org/w/api.php"

# .txt files under <current working directory>/assets/ (cwd is usually where Jupyter was started)
OUTPUT_FOLDER_NAME = "wikipedia_finance_india"
OUTPUT_DIR = Path.cwd() / "assets" / OUTPUT_FOLDER_NAME

# 0 = no cap on how many article titles to collect (full subtree except excluded branches)
MAX_ARTICLES = 10

# BFS category depth, counting the root as 0. None = no limit. 0 = only the root
# category. N = descend through depth N (root=0, first subcat tier=1, ...); 3 =
# full traversal to the 3rd subcategory level, subject to max articles / exclusions.
MAX_DEPTH: int | None = 3

REQUEST_DELAY_S = 0.35
EXCHARS = 80_000  # max extract length per page (prop=extracts)

USER_AGENT = (
    "WikipediaCategoryNotebook/1.0 (educational data prep; "
    "contact: local-user@example.invalid)"
)

## Implementation

In [ ]:
import html
import json
import re
import time
import unicodedata
import urllib.error
import urllib.parse
import urllib.request
from collections import deque
from dataclasses import dataclass, field
from pathlib import Path


def category_title(name: str) -> str:
    name = name.strip()
    if not name.lower().startswith("category:"):
        return "Category:" + name.replace(" ", "_")
    return "Category:" + name[9:].strip().replace(" ", "_")


def normalize_cat_key(title: str) -> str:
    """Stable key for comparing category titles from API vs user input."""
    t = title.strip()
    if t.lower().startswith("category:"):
        t = t[9:]
    t = t.replace("_", " ").strip().lower()
    return t


def excluded_category_keys(names: list[str]) -> set[str]:
    return {normalize_cat_key(category_title(n)) for n in names}


def safe_filename(title: str, max_len: int = 180) -> str:
    s = re.sub(r"[^\w\-_.]+", "_", title, flags=re.UNICODE)
    s = s.strip("_") or "untitled"
    return s[:max_len]


def api_get(params: dict[str, str]) -> dict:
    query = urllib.parse.urlencode(params)
    url = f"{API_URL}?{query}"
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    time.sleep(REQUEST_DELAY_S)
    with urllib.request.urlopen(req, timeout=60) as resp:
        return json.loads(resp.read().decode("utf-8"))


def paginate_category_members(
    cmtitle: str,
    *,
    cmnamespace: str,
    cmtype: str,
) -> list[dict]:
    members: list[dict] = []
    continue_token: str | None = None
    while True:
        params: dict[str, str] = {
            "action": "query",
            "format": "json",
            "list": "categorymembers",
            "cmtitle": cmtitle,
            "cmnamespace": cmnamespace,
            "cmtype": cmtype,
            "cmlimit": "50",
        }
        if continue_token:
            params["cmcontinue"] = continue_token
        data = api_get(params)
        batch = data.get("query", {}).get("categorymembers", [])
        members.extend(batch)
        cont = data.get("continue", {})
        continue_token = cont.get("cmcontinue")
        if not continue_token or not batch:
            break
    return members


@dataclass
class CrawlReport:
    root: str
    excluded_keys: set[str]
    max_depth: int | None
    categories_visited: int = 0
    subcats_skipped_excluded: list[str] = field(default_factory=list)
    article_titles: list[str] = field(default_factory=list)


def collect_article_titles_recursive_excluding(
    category_display: str,
    *,
    excluded_subcategory_names: list[str],
    max_articles: int | None,
    max_depth: int | None = None,
) -> CrawlReport:
    """BFS over the category tree; do not enter excluded subcategory branches.

    *max_depth*: ``None`` = no limit. ``0`` = only the root category. ``N`` >= 1
    = visit categories up to depth ``N`` (root depth is ``0``; children are
    ``depth + 1``). Article collection and subcategory following stop when the
    cap is hit.
    """
    root = category_title(category_display)
    excluded = excluded_category_keys(excluded_subcategory_names)
    report = CrawlReport(root=root, excluded_keys=excluded, max_depth=max_depth)

    seen_titles: set[str] = set()
    ordered: list[str] = []
    visited_categories: set[str] = set()
    q: deque[tuple[str, int]] = deque([(root, 0)])
    cap = None if max_articles in (None, 0) else int(max_articles)
    depth_unbounded = max_depth is None

    while q:
        cmtitle, depth = q.popleft()
        if cmtitle in visited_categories:
            continue
        visited_categories.add(cmtitle)

        for m in paginate_category_members(
            cmtitle, cmnamespace="0", cmtype="page"
        ):
            if cap is not None and len(ordered) >= cap:
                break
            t = m.get("title")
            if not t or t in seen_titles:
                continue
            seen_titles.add(t)
            ordered.append(t)

        if cap is not None and len(ordered) >= cap:
            break

        for sm in paginate_category_members(
            cmtitle, cmnamespace="14", cmtype="subcat"
        ):
            st = sm.get("title")
            if not st or st in visited_categories:
                continue
            if normalize_cat_key(st) in excluded:
                report.subcats_skipped_excluded.append(st)
                continue
            child_d = depth + 1
            if not depth_unbounded and child_d > int(max_depth):
                continue
            q.append((st, child_d))

    report.categories_visited = len(visited_categories)
    report.article_titles = ordered
    return report


def fetch_parse_plaintext(title: str) -> str:
    params = {
        "action": "parse",
        "format": "json",
        "page": title,
        "prop": "text",
        "formatversion": "2",
    }
    try:
        data = api_get(params)
    except urllib.error.HTTPError:
        return ""
    raw_html = data.get("parse", {}).get("text", "") or ""
    if not raw_html:
        return ""
    raw_html = re.sub(r"(?is)<script[^>]*>.*?</script>", "", raw_html)
    raw_html = re.sub(r"(?is)<style[^>]*>.*?</style>", "", raw_html)
    plain = re.sub(r"<[^>]+>", " ", raw_html)
    plain = html.unescape(plain)
    plain = re.sub(r"\s+", " ", plain).strip()
    return plain


def fetch_extract(title: str) -> tuple[str, str]:
    """Returns (resolved_title, plain_text)."""
    params: dict[str, str] = {
        "action": "query",
        "format": "json",
        "prop": "extracts",
        "redirects": "1",
        "explaintext": "1",
        "exsectionformat": "plain",
        "titles": title,
    }
    if EXCHARS > 0:
        params["exchars"] = str(min(EXCHARS, 1_000_000))
    data = api_get(params)
    pages = data.get("query", {}).get("pages", {})
    extract = ""
    resolved = title
    for _pid, page in pages.items():
        resolved = page.get("title", title)
        extract = page.get("extract") or ""
    if extract:
        return resolved, extract
    fb = fetch_parse_plaintext(title)
    return resolved, fb


def clean_article_text(text: str) -> str:
    """Normalize and lightly clean Wikipedia plain text for storage."""
    if not text:
        return ""
    t = unicodedata.normalize("NFC", text)
    t = t.replace("\r\n", "\n").replace("\r", "\n")
    t = re.sub(r"\n{3,}", "\n\n", t)
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n +", "\n", t)
    t = re.sub(r" +\n", "\n", t)
    return t.strip()


def write_article_txt(out_dir: Path, title: str, body: str) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / f"{safe_filename(title)}.txt"
    content = f"# {title}\n\n{clean_article_text(body)}\n"
    path.write_text(content, encoding="utf-8")
    return path

## Run crawl + download

In [ ]:
out_path = Path(OUTPUT_DIR).resolve()
excluded_keys = excluded_category_keys(EXCLUDED_SUBCATEGORIES)

print("Root:", category_title(ROOT_CATEGORY))
print("API:", API_URL)
print("Output:", out_path)
print("Excluded subcategory keys:", sorted(excluded_keys))
print(
    "Max category BFS depth:",
    "unlimited" if MAX_DEPTH is None else MAX_DEPTH,
    "(root=0; use None for unlimited, 0 for root-only)",
)

crawl = collect_article_titles_recursive_excluding(
    ROOT_CATEGORY,
    excluded_subcategory_names=EXCLUDED_SUBCATEGORIES,
    max_articles=MAX_ARTICLES,
    max_depth=MAX_DEPTH,
)

print(
    f"Max depth: {crawl.max_depth!r} | "
    f"Categories visited: {crawl.categories_visited} | "
    f"Article titles collected: {len(crawl.article_titles)}"
)
if crawl.subcats_skipped_excluded:
    uniq = sorted(set(crawl.subcats_skipped_excluded))
    print(f"Excluded branches skipped at enqueue time ({len(uniq)}):")
    for s in uniq:
        print("  -", s)

In [ ]:
download_rows: list[dict] = []

for i, t in enumerate(crawl.article_titles, start=1):
    resolved, raw = fetch_extract(t)
    cleaned = clean_article_text(raw)
    status = "ok" if cleaned else "empty_or_failed"
    path = None
    if cleaned:
        path = write_article_txt(out_path, resolved, cleaned)
    download_rows.append(
        {
            "requested_title": t,
            "resolved_title": resolved,
            "status": status,
            "chars": len(cleaned),
            "path": str(path) if path else "",
        }
    )
    if i % 10 == 0 or i == len(crawl.article_titles):
        print(f"Fetched {i}/{len(crawl.article_titles)} … last: {resolved!r} ({status})")

print("Done.")

## Report

In [ ]:
ok = [r for r in download_rows if r["status"] == "ok"]
bad = [r for r in download_rows if r["status"] != "ok"]
total_chars = sum(r["chars"] for r in ok)

print("=" * 72)
print("DOWNLOAD REPORT")
print("=" * 72)
print(f"Root category:        {crawl.root}")
print(f"Output directory:     {out_path}")
print(f"Categories visited:   {crawl.categories_visited}")
print(f"Titles targeted:      {len(download_rows)}")
print(f"Pages written (.txt): {len(ok)}")
print(f"Failed / empty:       {len(bad)}")
print(f"Total body chars:     {total_chars}")
print()
print("Excluded subcategories (not traversed):")
for name in EXCLUDED_SUBCATEGORIES:
    print(f"  - {name!r}  -> key {normalize_cat_key(category_title(name))!r}")
skipped = sorted(set(crawl.subcats_skipped_excluded))
print(f"\nSubcategory links skipped during BFS: {len(skipped)}")
for s in skipped:
    print("  ", s)
print()
if bad:
    print("Rows without saved text:")
    for r in bad:
        print(f"  - {r['requested_title']!r} -> {r['resolved_title']!r}")

In [ ]:
import pandas as pd
from IPython.display import display

display(pd.DataFrame(download_rows))